# 🚀 50M Bengali GPT - ২-স্টেজ প্রোডাকশন ট্রেনিং (Pretraining + SFT)
### 🧠 ২-স্টেজ আর্কিটেকচার:
1. **স্টেজ ১ (Pretraining - ভাষা শিক্ষা):** `corpus.txt` পড়ে মডেল ব্যাকরণ, শব্দভাণ্ডার ও বাক্যগঠন শেখে (২ ইপক, lr=3e-4)।
2. **স্টেজ ২ (SFT - প্রশ্ন-উত্তর শিক্ষা):** `sft_data.txt` পড়ে মডেল ইউজার থেকে প্রশ্ন পেয়ে সুনির্দিষ্ট উত্তর দেওয়া শেখে (১ ইপক, lr=1e-4)।

### ⚡ স্পেসিফিকেশন:
- **প্যারামিটার:** ~54.3 Million (১০০% আনফ্রোজেন, পুরো মডেল স্ক্র্যাচ থেকে শিখবে)
- **কনটেক্সট লেন্থ:** 512 Tokens (~৩০০-৩৫০ বাংলা শব্দ ধারণক্ষমতা)
- **ভোকাবুলারি:** 10,000 (ByteLevel BPE - বাংলা, ইংরেজি ও গণিত সমৃদ্ধ)
- **হার্ডওয়্যার:** Colab Free T4 GPU (~1.0 GB VRAM খরচ, 14 GB মেমরি নিরাপদ)

In [ ]:
# Step 1: GPU চেক করুন (NVIDIA T4 থাকা নিশ্চিত করুন)
!nvidia-smi

In [ ]:
# Step 2: Google Drive মাউন্ট করুন (উভয় স্টেজের চেকপয়েন্ট সেভ করার জন্য)
from google.colab import drive
import os

drive.mount('/content/drive')

DRIVE_BASE_DIR = '/content/drive/MyDrive/bengali_gpt_50m_checkpoints'
STAGE1_DIR = os.path.join(DRIVE_BASE_DIR, 'stage1_pretrain')
STAGE2_DIR = os.path.join(DRIVE_BASE_DIR, 'stage2_sft')
os.makedirs(STAGE1_DIR, exist_ok=True)
os.makedirs(STAGE2_DIR, exist_ok=True)
print(f"✓ Drive ডিরেক্টরি রেডি:\n  - Stage 1: {STAGE1_DIR}\n  - Stage 2: {STAGE2_DIR}")

In [ ]:
# Step 3: রিপোজিটরি ক্লোন ও লাইব্রেরি ইনস্টল করুন
import os

%cd /content
!rm -rf ss_100m
!git clone https://github.com/kajshikhi49-afk/ss_100m.git

%cd /content/ss_100m/ss_50million
!pip install -q tokenizers torch numpy datasets pyarrow
print("✓ এনভায়রনমেন্ট ও ডিপেনডেন্সি প্রস্তুত!")

In [ ]:
# Step 4: 📁 ডেটা প্রস্তুতকরণ (২টি ফাইল তৈরি: corpus.txt এবং sft_data.txt)
import os
import re
import json
import random
from datasets import load_dataset

os.makedirs('data', exist_ok=True)
corpus_path = 'data/corpus.txt'      # স্টেজ-১ এর জন্য
sft_data_path = 'data/sft_data.txt'  # স্টেজ-২ এর জন্য

print("⚡ ডেটাসেট তৈরি শুরু হচ্ছে (বাংলা + ইংরেজি + গণিত + প্রশ্ন-উত্তর)...")

corpus_lines = []
sft_lines = []

# ১. লোকাল বা ড্রাইভে থাকা jsonl ডেটা লোড (যদি থাকে)
jsonl_files = ['data/digital_marketing.jsonl', '/content/drive/MyDrive/digital_marketing.jsonl', 'data/nctb_data.jsonl']
for jf in jsonl_files:
    if os.path.exists(jf):
        print(f"  -> পাওয়া গেছে: {jf}")
        with open(jf, 'r', encoding='utf-8') as f:
            for line in f:
                try:
                    d = json.loads(line.strip())
                    inst = d.get('instruction', '').strip()
                    out = d.get('output', '').strip()
                    if inst and out:
                        # স্টেজ-১ এর জন্য: সাধারণ টেক্সট
                        corpus_lines.append(f"{inst} {out}")
                        # স্টেজ-২ এর জন্য: চ্যাট / প্রশ্ন-উত্তর ফরম্যাট
                        sft_lines.append(f"প্রশ্ন: {inst} উত্তর: {out}")
                except:
                    pass

# ২. উইকিপিডিয়া থেকে ভাষা শিক্ষার জন্য টেক্সট সংগ্রহ
print("  -> বাংলা ও ইংরেজি উইকিপিডিয়া সংগ্রহ হচ্ছে...")
wiki_bn = load_dataset('wikimedia/wikipedia', '20231101.bn', split='train', streaming=True)
bn_count = 0
for item in wiki_bn:
    for p in item.get('text', '').split('\n'):
        p = p.strip()
        if len(p) >= 30 and re.search(r'[\u0980-\u09FF]', p):
            corpus_lines.append(p)
            bn_count += 1
            if bn_count >= 150000: break
    if bn_count >= 150000: break

wiki_en = load_dataset('wikimedia/wikipedia', '20231101.en', split='train', streaming=True)
en_count = 0
for item in wiki_en:
    for p in item.get('text', '').split('\n'):
        p = p.strip()
        if len(p) >= 35 and re.search(r'[a-zA-Z]', p):
            corpus_lines.append(p)
            en_count += 1
            if en_count >= 40000: break
    if en_count >= 40000: break

# ৩. গণিত ও পাটিগণিত সমস্যা জেনারেশন (প্রশ্ন-উত্তর ফরম্যাট সহ)
print("  -> পাটিগণিত ও গণিত ডেটা প্রস্তুত হচ্ছে...")
random.seed(42)
for _ in range(15000):
    a, b = random.randint(2, 500), random.randint(2, 500)
    q_bn = f"{a} এর সাথে {b} যোগ করলে কত হয়?"
    a_bn = f"{a} + {b} = {a + b}।"
    corpus_lines.append(f"{q_bn} {a_bn}")
    sft_lines.append(f"প্রশ্ন: {q_bn} উত্তর: {a_bn}")
    
    m1, m2 = random.randint(2, 50), random.randint(2, 20)
    q_en = f"What is {m1} multiplied by {m2}?"
    a_en = f"{m1} * {m2} = {m1 * m2}."
    corpus_lines.append(f"{q_en} {a_en}")
    sft_lines.append(f"প্রশ্ন: {q_en} উত্তর: {a_en}")

# শাফলিং
random.shuffle(corpus_lines)
random.shuffle(sft_lines)

# ফাইলে সেভ
with open(corpus_path, 'w', encoding='utf-8') as f:
    for l in corpus_lines:
        f.write(l + '\n')

with open(sft_data_path, 'w', encoding='utf-8') as f:
    for l in sft_lines:
        f.write(l + '\n')

print("=" * 60)
print(f"✓ স্টেজ-১ ফাইল (corpus.txt): {len(corpus_lines):,} লাইন ({os.path.getsize(corpus_path)/(1024*1024):.1f} MB)")
print(f"✓ স্টেজ-২ ফাইল (sft_data.txt): {len(sft_lines):,} লাইন ({os.path.getsize(sft_data_path)/(1024*1024):.1f} MB)")
print("=" * 60)

In [ ]:
# Step 5: ⚡ সমন্বিত কর্পাস থেকে মাল্টিলিঙ্গুয়াল 10,000 Vocab BPE টোকেনাইজার তৈরি
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders

special_tokens = ['<PAD>', '<UNK>', '<BOS>', '<EOS>', '<|system|>', '<|user|>', '<|assistant|>', '<|math|>']
tok = Tokenizer(models.BPE(unk_token='<UNK>'))
tok.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False, use_regex=False)
tok.decoder = decoders.ByteLevel()

trainer = trainers.BpeTrainer(
    vocab_size=10000,
    special_tokens=special_tokens,
    min_frequency=2,
    show_progress=True
)

print("⚡ টোকেনাইজার ট্রেনিং শুরু হচ্ছে...")
tok.train(['data/corpus.txt', 'data/sft_data.txt'], trainer)
tok.save('tokenizer.json')
print(f"✓ টোকেনাইজার প্রস্তুত! Vocab Size: {tok.get_vocab_size():,}")

In [ ]:
# Step 6: 🚀 [স্টেজ ১] প্রি-ট্রেনিং (Pretraining - ভাষা শিক্ষা | ২ ইপক)
# মডেল স্ক্র্যাচ থেকে corpus.txt পড়ে ব্যাকরণ, বাক্য ও ভাষা শিখবে
import os
import sys
import time
import math
import torch
import torch.nn as nn
from torch.cuda.amp import autocast, GradScaler
from src.config import GPTConfig
from src.model import BengaliGPT as GPT
from src.dataset import BengaliDataset
from tokenizers import Tokenizer

device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True

# ১. ডেটাসেট লোড (corpus.txt)
tokenizer = Tokenizer.from_file("tokenizer.json")
dataset_stage1 = BengaliDataset(corpus_path="data/corpus.txt", tokenizer=tokenizer, block_size=GPTConfig.block_size, split_ratio=0.9)

# ২ ইপকের জন্য স্টেপ হিসাব
tokens_per_step = GPTConfig.batch_size * GPTConfig.gradient_accumulation_steps * GPTConfig.block_size
steps_per_epoch = dataset_stage1.train_len // tokens_per_step
STAGE1_EPOCHS = 2.0
stage1_max_iters = int(steps_per_epoch * STAGE1_EPOCHS)

print(f"✓ স্টেজ-১ লক্ষ্য: {STAGE1_EPOCHS} ইপক = {stage1_max_iters:,} স্টেপস")

# ২. মডেল ইনিশিয়ালাইজেশন
raw_model = GPT(GPTConfig).to(device)
optimizer = torch.optim.AdamW(raw_model.parameters(), lr=GPTConfig.learning_rate, betas=(0.9, 0.95), weight_decay=0.1)
scaler = GradScaler()

def get_lr(it, max_iters, lr=3e-4, min_lr=3e-5):
    warmup = 250
    if it < warmup: return lr * it / warmup
    if it > max_iters: return min_lr
    decay = (it - warmup) / (max_iters - warmup)
    return min_lr + 0.5 * (1.0 + math.cos(math.pi * decay)) * (lr - min_lr)

print("=" * 65)
print("🔥 [স্টেজ ১] প্রি-ট্রেনিং শুরু হচ্ছে (ভাষা শিক্ষা - ২ ইপক)... ")
print("=" * 65)

raw_model.train()
optimizer.zero_grad(set_to_none=True)
start_time = time.time()

for step in range(1, stage1_max_iters + 1):
    lr = get_lr(step, stage1_max_iters, lr=GPTConfig.learning_rate, min_lr=GPTConfig.min_lr)
    for param_group in optimizer.param_groups: param_group['lr'] = lr

    accum_loss = 0.0
    for micro in range(GPTConfig.gradient_accumulation_steps):
        x, y = dataset_stage1.get_batch('train', batch_size=GPTConfig.batch_size, device=device)
        with autocast(dtype=torch.float16):
            logits, loss = raw_model(x, y)
            loss = loss / GPTConfig.gradient_accumulation_steps
        scaler.scale(loss).backward()
        accum_loss += loss.item()

    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(raw_model.parameters(), 1.0)
    scaler.step(optimizer)
    scaler.update()
    optimizer.zero_grad(set_to_none=True)

    if step % 250 == 0 or step == 1:
        ep = (step * tokens_per_step) / dataset_stage1.train_len
        print(f"[Stage 1] Step {step:4d}/{stage1_max_iters} (Epoch {ep:.2f}) | Train Loss: {accum_loss:.4f} | LR: {lr:.2e}")

    if step % 500 == 0 or step == stage1_max_iters:
        ckpt = os.path.join(STAGE1_DIR, f"stage1_step_{step}.pt")
        torch.save(raw_model.state_dict(), ckpt)

# স্টেজ ১ ফাইনাল সেভ
stage1_final_path = os.path.join(DRIVE_BASE_DIR, "checkpoint_stage_1.pt")
torch.save(raw_model.state_dict(), stage1_final_path)
print(f"🎉 স্টেজ ১ সম্পন্ন! মডেল সেভ হয়েছে: {stage1_final_path}")

In [ ]:
# Step 7: 🎯 [স্টেজ ২] SFT ফাইন-টিউনিং (প্রশ্ন-উত্তর / চ্যাটবট শিক্ষা | ১ ইপক)
# স্টেজ ১ এর মডেল লোড করে sft_data.txt দিয়ে প্রশ্ন-উত্তরের নিয়ম শেখানো হবে
import os
import time
import math
import torch
import torch.nn as nn
from torch.cuda.amp import autocast, GradScaler
from src.config import GPTConfig
from src.model import BengaliGPT as GPT
from src.dataset import BengaliDataset
from tokenizers import Tokenizer

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# ১. স্টেজ ১ এর চেকপয়েন্ট লোড
stage1_final_path = os.path.join(DRIVE_BASE_DIR, "checkpoint_stage_1.pt")
if not os.path.exists(stage1_final_path):
    raise FileNotFoundError(f"স্টেজ ১ এর ফাইল পাওয়া যায়নি: {stage1_final_path}। অনুগ্রহ করে আগে Step 6 চালান।")

tokenizer = Tokenizer.from_file("tokenizer.json")
sft_model = GPT(GPTConfig).to(device)
sft_model.load_state_dict(torch.load(stage1_final_path, map_location=device))
print(f"✓ স্টেজ ১ এর ভাষা-মডেল সফলভাবে লোড হয়েছে: {stage1_final_path}")

# ২. স্টেজ ২ ডেটাসেট লোড (sft_data.txt)
dataset_stage2 = BengaliDataset(corpus_path="data/sft_data.txt", tokenizer=tokenizer, block_size=GPTConfig.block_size, split_ratio=0.9)

tokens_per_step = GPTConfig.batch_size * GPTConfig.gradient_accumulation_steps * GPTConfig.block_size
steps_per_epoch = dataset_stage2.train_len // tokens_per_step
STAGE2_EPOCHS = 1.0  # প্রশ্ন-উত্তরের জন্য ১ ইপকই আদর্শ
stage2_max_iters = int(steps_per_epoch * STAGE2_EPOCHS)

print(f"✓ স্টেজ-২ লক্ষ্য: {STAGE2_EPOCHS} ইপক = {stage2_max_iters:,} স্টেপস")

# ৩. অপটিমাইজার (লোয়ার লার্নিং রেট 1e-4)
optimizer = torch.optim.AdamW(sft_model.parameters(), lr=GPTConfig.sft_learning_rate, betas=(0.9, 0.95), weight_decay=0.1)
scaler = GradScaler()

def get_sft_lr(it, max_iters, lr=1e-4, min_lr=1e-5):
    warmup = 100
    if it < warmup: return lr * it / warmup
    if it > max_iters: return min_lr
    decay = (it - warmup) / (max_iters - warmup)
    return min_lr + 0.5 * (1.0 + math.cos(math.pi * decay)) * (lr - min_lr)

print("=" * 65)
print("🔥 [স্টেজ ২] SFT চ্যাটবট ট্রেনিং শুরু হচ্ছে (প্রশ্ন-উত্তর শিক্ষা - ১ ইপক)... ")
print("=" * 65)

sft_model.train()
optimizer.zero_grad(set_to_none=True)
start_time = time.time()

for step in range(1, stage2_max_iters + 1):
    lr = get_sft_lr(step, stage2_max_iters, lr=GPTConfig.sft_learning_rate, min_lr=GPTConfig.sft_min_lr)
    for param_group in optimizer.param_groups: param_group['lr'] = lr

    accum_loss = 0.0
    for micro in range(GPTConfig.gradient_accumulation_steps):
        x, y = dataset_stage2.get_batch('train', batch_size=GPTConfig.batch_size, device=device)
        with autocast(dtype=torch.float16):
            logits, loss = sft_model(x, y)
            loss = loss / GPTConfig.gradient_accumulation_steps
        scaler.scale(loss).backward()
        accum_loss += loss.item()

    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(sft_model.parameters(), 1.0)
    scaler.step(optimizer)
    scaler.update()
    optimizer.zero_grad(set_to_none=True)

    if step % 250 == 0 or step == 1:
        ep = (step * tokens_per_step) / dataset_stage2.train_len
        print(f"[Stage 2] Step {step:4d}/{stage2_max_iters} (Epoch {ep:.2f}) | SFT Loss: {accum_loss:.4f} | LR: {lr:.2e}")

    if step % 500 == 0 or step == stage2_max_iters:
        ckpt = os.path.join(STAGE2_DIR, f"stage2_step_{step}.pt")
        torch.save(sft_model.state_dict(), ckpt)

# চূড়ান্ত প্রোডাকশন মডেল সেভ
final_model_path = os.path.join(DRIVE_BASE_DIR, "checkpoint_stage_2_final.pt")
torch.save(sft_model.state_dict(), final_model_path)
print(f"🎉 অভিনন্দন! চূড়ান্ত ২-স্টেজ প্রোডাকশন মডেল প্রস্তুত: {final_model_path}")

In [ ]:
# Step 8: 💬 চ্যাটবট টেস্ট সেল (প্রশ্ন করুন, মডেল উত্তর দেবে!)
final_model_path = os.path.join(DRIVE_BASE_DIR, "checkpoint_stage_2_final.pt")
chat_model = GPT(GPTConfig).to(device)
chat_model.load_state_dict(torch.load(final_model_path, map_location=device))
chat_model.eval()
tokenizer = Tokenizer.from_file("tokenizer.json")

# এখানে আপনার প্রশ্ন লিখুন:
user_question = "ডিজিটাল মার্কেটিং কী?"

prompt = f"প্রশ্ন: {user_question} উত্তর:"
enc = tokenizer.encode(prompt)
ids = enc.ids if hasattr(enc, 'ids') else enc
input_tensor = torch.tensor([ids], dtype=torch.long, device=device)

with torch.no_grad():
    out = chat_model.generate(
        input_tensor,
        max_new_tokens=150,
        temperature=0.7,
        top_k=40,
        repetition_penalty=1.25
    )

reply = tokenizer.decode(out[0].cpu().tolist())
print("=" * 60)
print(reply)
print("=" * 60)